# PySersic

`PySersic_Fitter` fits Sersic and/or point-source profiles to galaxy
cutouts using [`pysersic`](https://github.com/pysersic/pysersic), a
JAX/numpyro Bayesian Sersic-fitting code. It is the `pysersic`-backed
counterpart to `Galfit_Fitter`, sharing the same `Morphology_Fitter`
interface, but with genuine Bayesian priors instead of GALFIT's
text-file box constraints.

This notebook demonstrates the API on synthetic postage stamps (no
real survey data needed), focusing on the feature this fitter adds:
**straightforward prior customization** via `primary_priors`/
`neighbour_priors` dicts, plus the three available `model`s and
simultaneous neighbour fitting.

.. note::
   Requires the optional `pysersic` dependency group:
   ``pip install galfind[pysersic]``.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from galfind.properties.PySersic import PySersic_Fitter


def make_sersic_stamp(ny=41, nx=41, amp=10.0, scale=3.0, noise=0.05, seed=0):
    """A synthetic n=1 (exponential) Sersic profile + noise, for a
    single unresolved-PSF pixel (i.e. PSF = delta function -- no
    convolution smoothing needed for this illustrative example)."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[:ny, :nx]
    y0, x0 = ny / 2, nx / 2
    r = np.sqrt((xx - x0) ** 2 + (yy - y0) ** 2)
    sci = amp * np.exp(-r / scale) + rng.normal(scale=noise, size=(ny, nx))
    rms = np.full((ny, nx), noise, dtype=np.float32)
    mask = np.zeros((ny, nx), dtype=bool)
    psf = np.zeros((ny, nx), dtype=np.float32)
    psf[ny // 2, nx // 2] = 1.0
    return sci.astype(np.float32), rms, mask, psf


sci, rms, mask, psf = make_sersic_stamp()
plt.imshow(sci, origin="lower")
plt.title("Synthetic n=1 Sersic stamp")
plt.colorbar()
plt.show()


## A free fit

With no `primary_priors`, every parameter (`n`, `r_eff`, `ellip`,
`theta`, `flux`, position) is fitted freely, using `pysersic`'s own
default priors.

In [ ]:
fitter = PySersic_Fitter(psf=None, model="sersic", method="laplace")
map_params, idata = fitter._fit_array(sci, rms, mask, psf)
print(f"n     = {map_params['n'][1]:.2f}  (16/84: {map_params['n'][0]:.2f} / {map_params['n'][2]:.2f})")
print(f"r_eff = {map_params['r_eff'][1]:.2f} pix")


## Holding a parameter fixed

Passing a single number for a parameter -- rather than a `[lo, hi]`
range -- holds it fixed at exactly that value for every posterior draw
(e.g. `n=1` for a pure exponential disk). This can't be done with a
plain `numpyro.distributions.Delta` prior (numpyro's SVI/MCMC
initialization explicitly forbids sampling a bare `Delta` site), so
`PySersic_Fitter` removes the parameter from the sampled priors
entirely and injects it back in as a `numpyro.deterministic` constant
at render time -- see `PySersic_Fitter._patch_fixed_params_build_model`
for the full mechanism.

In [ ]:
fixed_fitter = PySersic_Fitter(
    psf=None, model="sersic",
    primary_priors={"sersic": {"n": 1.0}},
    method="laplace",
)
map_params_fixed, _ = fixed_fitter._fit_array(sci, rms, mask, psf)
n_lo, n_med, n_hi = map_params_fixed["n"]
print(f"n = {n_med} (16th={n_lo}, 84th={n_hi}) -- exactly fixed, zero posterior width")


## A flat/uniform prior bound

Passing a `[lo, hi]` pair replaces `pysersic`'s default prior for that
parameter with a flat/uniform prior over that range -- useful for
capping `r_eff` when an unbounded fit would run away to an
unphysically large size (e.g. for a broad, low-contrast source).

In [ ]:
broad_sci, broad_rms, broad_mask, broad_psf = make_sersic_stamp(
    ny=61, nx=61, amp=1.0, scale=15.0, noise=0.05, seed=1,
)
capped_fitter = PySersic_Fitter(
    psf=None, model="sersic",
    primary_priors={"sersic": {"r_eff": [0.5, 10.0]}},
    method="laplace",
)
capped_params, _ = capped_fitter._fit_array(broad_sci, broad_rms, broad_mask, broad_psf)
print(f"r_eff = {capped_params['r_eff'][1]:.2f} pix (capped at 10)")


## The three available models

- `"sersic"` -- a pure Sersic host (used above).
- `"pointsource"` -- a pure unresolved point source (just position +
  flux; rendered directly from the PSF).
- `"sersic_pointsource"` -- a Sersic host *plus* an unresolved nuclear
  point source, with a fitted point-source flux fraction `f_ps`
  (useful for e.g. decomposing an AGN point source from its host
  galaxy).

In [ ]:
ny, nx = 21, 21
rng = np.random.default_rng(2)
point_psf = np.zeros((ny, nx), dtype=np.float32)
point_psf[ny // 2, nx // 2] = 1.0
point_sci = (10.0 * point_psf + rng.normal(scale=0.05, size=(ny, nx))).astype(np.float32)
point_rms = np.full((ny, nx), 0.05, dtype=np.float32)
point_mask = np.zeros((ny, nx), dtype=bool)

ps_fitter = PySersic_Fitter(psf=None, model="pointsource", method="laplace")
ps_params, _ = ps_fitter._fit_array(point_sci, point_rms, point_mask, point_psf)
print(f"flux = {ps_params['flux'][1]:.2f}")

sci_ps = sci.copy()
sci_ps[sci_ps.shape[0] // 2, sci_ps.shape[1] // 2] += 5.0
sersic_ps_fitter = PySersic_Fitter(psf=None, model="sersic_pointsource", method="laplace")
sersic_ps_params, _ = sersic_ps_fitter._fit_array(sci_ps, rms, mask, psf)
print(f"f_ps = {sersic_ps_params['f_ps'][1]:.3f}")


## Neighbour fitting

Just like `Galfit_Fitter`'s `neighbours_model`/`neighbour_constraints`,
`PySersic_Fitter` can fit a detected neighbour *simultaneously* with the
primary source (via `pysersic.FitMulti`), which avoids the neighbour's
flux biasing the primary source's fitted size the way an
ignore-the-neighbour single-source fit would. When wired up to a real
`Catalogue`, the neighbour catalogue is built automatically from
`_get_neighbours` (shared with `Galfit_Fitter`); here we build one by
hand for two synthetic overlapping sources.

In [ ]:
ny, nx = 51, 51
rng = np.random.default_rng(4)
yy, xx = np.mgrid[:ny, :nx]


def sersic_blob(x0, y0, amp, scale):
    r = np.sqrt((xx - x0) ** 2 + (yy - y0) ** 2)
    return amp * np.exp(-r / scale)


primary_x, primary_y = nx / 2, ny / 2
neighbour_x, neighbour_y = primary_x + 12, primary_y + 4
sci2 = (
    sersic_blob(primary_x, primary_y, 10.0, 3.0)
    + sersic_blob(neighbour_x, neighbour_y, 8.0, 2.5)
    + rng.normal(scale=0.05, size=(ny, nx))
).astype(np.float32)
rms2 = np.full((ny, nx), 0.05, dtype=np.float32)
mask2 = np.zeros((ny, nx), dtype=bool)
psf2 = np.zeros((ny, nx), dtype=np.float32)
psf2[ny // 2, nx // 2] = 1.0

plt.imshow(sci2, origin="lower")
plt.title("Primary + neighbour")
plt.colorbar()
plt.show()

single_fitter = PySersic_Fitter(psf=None, model="sersic", method="laplace")
single_params, _ = single_fitter._fit_array(sci2, rms2, mask2, psf2)

multi_fitter = PySersic_Fitter(psf=None, model="sersic", neighbours_model="sersic", method="laplace")
neighbour_catalog = {
    "x": np.array([primary_x, neighbour_x]),
    "y": np.array([primary_y, neighbour_y]),
    "flux": np.array([100.0, 60.0]),
    "r": np.array([3.0, 2.5]),
    "type": ["sersic", "sersic"],
}
multi_params, _ = multi_fitter._fit_array(
    sci2, rms2, mask2, psf2, neighbour_catalog=neighbour_catalog,
)

true_r_eff = 1.678 * 3.0  # b_1(n=1) * scale
print(f"true r_eff             ~= {true_r_eff:.2f} pix")
print(f"single-source fit r_eff = {single_params['r_eff'][1]:.2f} pix (biased by the neighbour's flux)")
print(f"joint fit primary r_eff = {multi_params['r_eff_0'][1]:.2f} pix (neighbour fitted separately as r_eff_1={multi_params['r_eff_1'][1]:.2f})")


## A real example: GS-z14-1

The synthetic examples above show the API mechanics; here's a real fit,
using the same pattern, on a real spectroscopically-confirmed z~14.3
galaxy from Carniani+24 (JADES), fit in NIRCam/F444W with `n` held
fixed at 1.

In [ ]:
import sys
sys.path.append("/nvme/scratch/work/austind/EPOCHS-v2/scripts")

import astropy.units as u
from galfind import config as galfind_config
from galfind.galaxy import Galaxy
from config import survey_aliases, survey_versions, min_flux_pc_err
from load_cat import full_data_load

survey = "JADES-DR3-GS-South"  # GS-z14-1 (Carniani+24)
ID = 30479
version = survey_versions.get(survey) or survey_versions.get(survey_aliases.get(survey, survey))

data = full_data_load(survey, version)
gal = Galaxy.from_data_id(
    data, ID,
    load_phot_kwargs={"ZP": u.Jy.to(u.ABmag), "min_flux_pc_err": min_flux_pc_err},
)
band_data = data.native["F444W"]

real_fitter = PySersic_Fitter(
    psf=band_data.psf, model="sersic",
    primary_priors={"sersic": {"n": 1.0}},
    method="svi-mvn",
)
cutout = gal.make_band_cutout(band_data, cutout_size=1.5 * u.arcsec)
# Saves/loads the fit under GALFIND_WORK/PySersic, following the same
# per-source directory convention `PySersic_Fitter._fit_cat` builds by
# default -- re-running this cell reloads the cached .h5 rather than
# re-fitting.
out_dir = f"{galfind_config['PYSERSIC']['OUTPUT_DIR']}/{version}/{survey}/F444W/{ID}"
result = real_fitter._fit_cutout(cutout, cat=None, plot=True, out_dir=out_dir)
print({k: float(v.value) for k, v in result.properties.items()})
print(f"reduced chi2 = {result.red_chi2:.2f}")


### Image / model / residual, and posterior corner plot

`PySersic_Result.plot()` shows the science image, best-fit model, and
residual (same 3-panel layout `Galfit_Result.plot()` uses), plus the
full posterior corner plot for every fitted parameter -- `n` shows up
as an exactly zero-width column since it was held fixed above.

In [ ]:
result.plot(save=False, show=True)


### Caching under `GALFIND_WORK`

`_fit_cat` (and, as here, `_fit_cutout` when given an explicit
`out_dir`) saves every fit's posterior samples/model image to an
`.h5` file under `out_dir`, keyed by the fitter's model + fixed/bounded
priors + neighbour-inclusion, e.g. under
`GALFIND_WORK/PySersic/output/<version>/<survey>/<filter>/<ID>/...`
when `out_dir` is left at `_fit_cat`'s own default. Calling
`_fit_cutout` again against the *same* `out_dir` reloads that cached
fit instead of re-running SVI -- useful since a real fit can take a
couple of minutes, but a re-load is near-instant.

In [ ]:
import time

t0 = time.time()
cached_result = real_fitter._fit_cutout(cutout, cat=None, plot=False, out_dir=out_dir)
print(f"reload took {time.time() - t0:.2f}s")
print(f"r_eff matches original fit: {cached_result.properties['r_eff'] == result.properties['r_eff']}")
